# XAI Evaluation: Reproducible Pipeline

This notebook executes the project's benchmark pipeline in small, inspectable steps. Metric values are conditional on the displayed protocol; they are not universal explanation-quality scores.

## 1. Set Up Environment and Inputs

In [ ]:
from pathlib import Path
import sys
import yaml
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
CONFIG_PATH = ROOT / "experiments/config.yaml"
OUTPUT_PATH = ROOT / "results/raw/notebook_benchmark.csv"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config

## 2. Define Plan Milestones as Executable Tasks

Each task declares dependencies and an output key. The runner below executes them in order and fails if a dependency is missing.

In [ ]:
tasks = [
    {"name": "load_data", "depends_on": [], "output": "data"},
    {"name": "fit_models", "depends_on": ["data"], "output": "models"},
    {"name": "evaluate_explanations", "depends_on": ["data", "models"], "output": "results"},
    {"name": "export", "depends_on": ["results"], "output": "csv"},
]
tasks

## 3. Implement Core Functions per Milestone

The package owns data loading, model definitions, explanations, and metric formulas. This notebook composes those functions and keeps experiment parameters visible.

In [ ]:
from xai_eval.data import breast_cancer_split
from xai_eval.models import make_models, positive_probability
from xai_eval.explainers import local_perturbation_attribution
from xai_eval.metrics import deletion_auc, comprehensiveness, sufficiency, max_sensitivity


def execute_task(task, state):
    """Execute one named milestone, validating dependency outputs first."""
    missing = [key for key in task["depends_on"] if key not in state]
    if missing:
        raise RuntimeError(f"{task['name']} is missing inputs: {missing}")
    name = task["name"]
    if name == "load_data":
        x_train, x_test, y_train, y_test, feature_names = breast_cancer_split(random_state=config["seed"])
        return {"data": (x_train, x_test, y_train, y_test, feature_names)}
    if name == "fit_models":
        x_train, _, y_train, _, _ = state["data"]
        fitted = make_models(random_state=config["seed"])
        for model in fitted.values():
            model.fit(x_train, y_train)
        return {"models": fitted}
    if name == "evaluate_explanations":
        return {"results": evaluate_benchmark(state["data"], state["models"])}
    if name == "export":
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        state["results"].to_csv(OUTPUT_PATH, index=False)
        return {"csv": OUTPUT_PATH}
    raise ValueError(f"Unknown task: {name}")


def evaluate_benchmark(data, models):
    x_train, x_test, y_train, y_test, _ = data
    baseline = np.median(x_train, axis=0)
    settings = config["benchmark"]
    records = []
    for model_name, model in models.items():
        for instance_index in range(min(settings["n_instances"], len(x_test))):
            x = x_test[instance_index]
            score_fn = lambda candidate, fitted=model: positive_probability(fitted, candidate)
            attribution = local_perturbation_attribution(model, x, baseline=baseline)
            top = np.argsort(-np.abs(attribution))[:max(1, int(0.2 * len(x)))]
            candidates = x[None, :] + np.eye(len(x))[:5] * settings["sensitivity_radius"]
            records.append({
                "model": model_name,
                "instance": instance_index,
                "deletion_auc": deletion_auc(score_fn, x, attribution, baseline=baseline,
                                             steps=settings["deletion_steps"]),
                "comprehensiveness": comprehensiveness(score_fn, x, top, baseline=baseline),
                "sufficiency": sufficiency(score_fn, x, top, baseline=baseline),
                "max_sensitivity": max_sensitivity(
                    lambda point: local_perturbation_attribution(model, point, baseline=baseline),
                    x, candidates, radius=settings["sensitivity_radius"]
                ),
            })
    return pd.DataFrame(records)

## 4. Run Task Pipeline End-to-End

Tasks execute in their declared order. Each output is placed in shared state for dependent tasks.

In [ ]:
state = {}
for task in tasks:
    try:
        state.update(execute_task(task, state))
        print(f"Completed {task['name']}: {task['output']}")
    except Exception as error:
        raise RuntimeError(f"Pipeline stopped at milestone {task['name']}") from error

results = state["results"]
results.head()

## 5. Validate Outputs with Assertions and Unit-Style Checks

These checks verify table structure, finite values, metric ranges, and the expected direction of a known linear-model deletion example.

In [ ]:
required_columns = {"model", "instance", "deletion_auc", "comprehensiveness", "sufficiency", "max_sensitivity"}
assert required_columns.issubset(results.columns)
assert len(results) == 3 * min(config["benchmark"]["n_instances"], len(state["data"][1]))
assert np.isfinite(results.select_dtypes(include="number").to_numpy()).all()
assert (results["max_sensitivity"] >= 0).all()

from xai_eval.metrics import deletion_auc
linear = lambda point: float(np.dot(np.array([1.0, 2.0, 0.0]), point))
example_x = np.array([1.0, 1.0, 4.0])
good = np.array([1.0, 2.0, 0.0])
random = np.array([0.0, 0.0, 1.0])
assert deletion_auc(linear, example_x, good, baseline=0.0, steps=3) < deletion_auc(
    linear, example_x, random, baseline=0.0, steps=3
)
print("Pipeline and metric invariants passed.")

## 6. Log Results and Export Artifacts

Summarize observed per-instance metrics and export them under `results/raw/`. Interpret ranking differences only within the configured model, data, baseline, and perturbation protocol.

In [ ]:
metrics = ["deletion_auc", "comprehensiveness", "sufficiency", "max_sensitivity"]
summary = results.groupby("model")[metrics].agg(["mean", "std"])
summary
